# Customer Churn Prediction — End-to-End Walkthrough

A complete churn-modelling workflow on the **IBM Telco Customer Churn** dataset. This notebook shows the full data-science process: exploratory analysis with [`eda-kit`](https://github.com/sahindemirbas/eda-kit), preprocessing, modelling with a Random Forest, and interpretation of what actually drives churn.

> The same steps run in one place via `python src/train_model.py`.

## Business context

Churn (customers leaving) is one of the most expensive problems a subscription business faces. **Acquiring a new customer costs far more than keeping an existing one**, so identifying who is likely to churn — *before* they leave — lets a company target retention offers efficiently. The goal here is to predict churn and, just as importantly, to understand *why* customers churn so the business can act.

In [ ]:
import pandas as pd
import eda_kit as ek

DATA_URL = (
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
    "master/data/Telco-Customer-Churn.csv"
)
df = pd.read_csv(DATA_URL)
df.head()

## 1. Data overview
Let eda-kit give us the full structural picture at once.

In [ ]:
ek.check_df(df)

## 2. Know the columns
Classify columns into categorical / numerical / cardinal before touching anything.

In [ ]:
cat_cols, num_cols, card_cols = ek.grab_col_names(df)
print("categorical:", cat_cols)
print("numerical:", num_cols)
print("cardinal (drop):", card_cols)

## 3. Target distribution
Churn is imbalanced (~26% churn) — that's why accuracy alone is a misleading metric here and why we use ROC-AUC and look at the baseline.

In [ ]:
print(df['Churn'].value_counts(normalize=True).round(3))
df['Churn'].value_counts().plot(kind='bar', title='Churn distribution', color=['#4c72b0','#c44e52'])

## 4. Missing values

In [ ]:
na_cols = ek.missing_values_table(df, na_name=True)
print(na_cols)

## 5. Cleanup: TotalCharges coercion
`TotalCharges` is stored as text with a few blanks. We coerce to numeric and drop the ~11 rows that are new customers with no total yet.

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges']).reset_index(drop=True)
print('rows after clean:', len(df))

# refresh column types after the dtype change
cat_cols, num_cols, card_cols = ek.grab_col_names(df)

## 6. Outliers — cap, don't drop

In [ ]:
for col in num_cols:
    if ek.check_outlier(df, col):
        print(f'  {col}: outliers -> capping')
        ek.replace_with_thresholds(df, col)
    else:
        print(f'  {col}: clean')

## 7. Encode categoricals
Collapse rare classes first, then one-hot encode (keeping the target out of encoding).

In [ ]:
encode_cols = [c for c in cat_cols if c != 'Churn']
df = ek.rare_encoder(df, rare_perc=0.01)
df = ek.one_hot_encoder(df, encode_cols)
print('encoded shape:', df.shape)

# drop high-cardinality / ID columns before modelling
df = df.drop(columns=card_cols)
print('after dropping cardinal cols:', df.shape)

## 8. Train / test split (stratified)

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('Churn', axis=1)
y = (df['Churn'] == 'Yes').astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('train:', X_tr.shape, '| test:', X_te.shape)
print('churn rate train/test:', round(y_tr.mean(),3), round(y_te.mean(),3))

## 9. Baselines first
A **dummy** (always predict the majority class) establishes the floor. Any real model must beat it — especially on ROC-AUC, where the dummy sits at 0.5.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def evaluate(model, X_te, y_te):
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]
    return {
        'accuracy': accuracy_score(y_te, y_pred),
        'precision': precision_score(y_te, y_pred),
        'recall': recall_score(y_te, y_pred),
        'f1': f1_score(y_te, y_pred),
        'roc_auc': roc_auc_score(y_te, y_proba),
    }

base = DummyClassifier(strategy='most_frequent').fit(X_tr, y_tr)
print('Baseline:', evaluate(base, X_te, y_te))

## 10. Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=5, random_state=42, n_jobs=-1)
model.fit(X_tr, y_tr)
metrics = evaluate(model, X_te, y_te)
print('RandomForest:', {k: round(v,4) for k,v in metrics.items()})

## 11. Confusion matrix

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_te, model.predict(X_te))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix')
plt.show()

## 12. What drives churn?
The most important features are business-readable: **tenure**, **total & monthly charges**, **fiber-optic internet**, **short contracts** and **electronic-check payment** stand out — the classic profile of a higher-value, month-to-month customer at risk.

In [ ]:
ek.plot_importance(model, X_tr, num=12)

## 13. Business takeaways

- **Tenure is the strongest signal** — churn concentrates in the first months; early onboarding / loyalty incentives pay off.
- **Month-to-month + electronic check** is a high-risk combination → push annual contracts and autopay.
- **Fiber-optic users churn more** → likely a service-quality / price perception issue worth investigating (not just "sell fiber").
- **Retention should target *before* customers become high spenders then leave** — use predicted churn probability as a prioritization score for CRM outreach.

> These are exactly the kind of insights a data analyst turns into a dashboard and a retention campaign, not just a model report.